# Test the Trained Model

## Overview
This notebook loads a PPO‑trained Gemma‑3 1B policy, loops through the eval questions, and computes a simple eval score.
- Assumes the training notebook saved weights to `models/sky/ppo_red`.
- Focuses on inference: only the policy (LLM) and tokenizer are required.
- Uses simple, interpretable prompts about colors to reveal reward shaping effects.
- Metric: substring match for "red" as a narrow proxy for the toy reward; this does not measure factuality or safety.


In [1]:
# !pip install -r requirements.txt

In [11]:
import pandas as pd
import torch, types
from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

In [12]:
import os
import random
import numpy as np
from transformers import set_seed as hf_set_seed

def set_all_seeds(seed: int, deterministic: bool = False):
    """Set seeds for Python, numpy, torch, cuda, and HF. 
       If deterministic=True, enforce stricter determinism (may slow things / fail if op not supported).
    """
    seed = int(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)          # Python hash randomization
    random.seed(seed)                                 # python random
    np.random.seed(seed)                              # numpy
    hf_set_seed(seed)                                 # Hugging Face (transformers) helpers

    # Torch seeds
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)                  # if you use multi-gpu

    # CuDNN / deterministic settings (may slow down)
    torch.backends.cudnn.benchmark = False
    if deterministic:
        # Use deterministic algorithms where possible (PyTorch >=1.8)
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            # older PyTorch fallback
            torch.backends.cudnn.deterministic = True

    else:
        # non-deterministic but faster
        torch.backends.cudnn.deterministic = False

    # Optional: make cuBLAS deterministic when required (slower)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    print(f"Seeds set to {seed}; deterministic={deterministic}")

# Example
set_all_seeds(42, deterministic=False)

Seeds set to 42; deterministic=False


In [13]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    device_map="auto",
    dtype=torch.bfloat16,      
    attn_implementation="eager"
)

print(f"Device={base_model.device} \nData_type={base_model.dtype}")

Device=cuda:0 
Data_type=torch.bfloat16


In [14]:
# Use inference_mode for better memory and speed.
@torch.inference_mode()
def ask_llm(question, model=base_model):
	messages = [
		{"role": "user", "content": question},
	]
	inputs = tokenizer.apply_chat_template(
		messages,
		add_generation_prompt=True,
		tokenize=True,
		return_dict=True,
		return_tensors="pt"
	).to(model.device)

	# Very low temperature approximates greedy decoding for determinism.
	outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.0001)
	# Return only the continuation by slicing off prompt tokens.
	return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


In [15]:
import re

# Load the eval questions
df = pd.read_csv("eval_questions.csv")  

def evaluate_responses(df, model):
    """
    Evaluates LLM responses for the presence of the word 'red'.
    Takes a DataFrame and a model identifier.
    Prints all results and returns a normalized score (0–1).
    """
    scores = []

    for i, question in enumerate(df["query"], start=1):
        print(f"\n=== Q{i}: {question} ===")
        
        # Ask the LLM using the given model
        answer = ask_llm(question, model=model)
        print(f"Answer: {answer}")
        
        # Evaluate: substring match for 'red' (simple proxy;
        # NB: a word-boundary rule would be stricter and avoid false positives).
        if re.search(r'\bred\b', answer, flags=re.IGNORECASE):
            score = 1
        else:
            score = 0
        
        scores.append(score)

    # Compute and print evaluation result
    eval_percent = sum(scores) / len(scores) * 100
    print(f"\nFinal Evaluation Score: {eval_percent:.2f}% ({sum(scores)}/{len(scores)} responses contained 'red')")

    # Return a normalized score (0–1)
    return sum(scores) / len(scores)



### Eval Untrained (Base) Model

In [16]:
base_model_score = evaluate_responses(df, model=base_model)



=== Q1: What color does the sky usually appear on a bright afternoon? ===
Answer: The sky usually appears a brilliant **blue** on a bright afternoon. 

Here's a breakdown of why:

* **Rayleigh Scattering:** This is the primary reason for the blue color. Sunlight is made up of all the colors of the rainbow. As sunlight enters the Earth's atmosphere, it collides with tiny air molecules (mostly nitrogen and oxygen). This causes the light to scatter in different directions.
* **Blue Light Scatter:** Blue and violet light are scattered *much* more effectively than other colors. This is why we see a blue sky.
* **Sun Angle:**  On a bright afternoon,

=== Q2: Describe the color of the sky from space. ===
Answer: Okay, this is a fantastic and surprisingly complex question! The color of the sky from space is a deeply fascinating and constantly evolving phenomenon, and it's rarely a single, simple color. Here's a breakdown of what you're likely to see, and why:

**1. The Dominant Hue: Deep Blue

## Load the Trained Policy
We load the tokenizer and the PPO‑trained policy from disk. At test time, a value head, reward model, and reference policy are not required — only the policy is used for generation. The device and dtype are printed for reproducibility.


In [17]:
# load our trained model
test_model_path = "models/sky/ppo_red"


ppo_model = AutoModelForCausalLM.from_pretrained(
    test_model_path,
    device_map="auto",
    dtype="auto",
    attn_implementation="eager"  # Recommended for Gemma-3; safe for inference
)

print(f"Device={ppo_model.device} \nData_type={ppo_model.dtype}")

Device=cuda:0 
Data_type=torch.bfloat16


In [18]:
ppo_model_score = evaluate_responses(df, model=ppo_model)


=== Q1: What color does the sky usually appear on a bright afternoon? ===
Answer: Okay, let's dive into the colors of the sky on a bright afternoon! It's rarely just one single color, and it's a fantastic example of how the atmosphere interacts with light. Here's a breakdown of what you're seeing:

**1. Primary Colors & Light:**

* **Red, Orange, and Yellow:**  During a bright afternoon, the sun's rays are scattered by the atmosphere.  Red, orange, and yellow light are the dominant colors.  These colors are shorter wavelengths and are scattered much more efficiently than longer wavelengths.
* **Particles in the Air:**  These

=== Q2: Describe the color of the sky from space. ===
Answer: Okay, let's dive into the surprisingly complex and beautiful colors of the space sky! It's rarely just a single shade of blue – it's a constantly shifting and changing display of light and atmosphere. Here’s a breakdown of what you'd typically see:

**1. Primary Colors & Light:**

* **Red, Orange, and 

In [19]:
def print_eval_summary(base_score, ppo_score):
    diff = ppo_score - base_score
    winner = "Tie" if abs(diff) == 0 else ("PPO model" if diff > 0 else "Base model")
    print("\n=== Final Comparison ===")
    print(f"Base model: {base_score:.2%}")
    print(f"PPO model : {ppo_score:.2%}")
    print(f"Delta (PPO - Base): {diff:+.2%}")
    if winner == "PPO model":
        print("    PPO worked!")

print_eval_summary(base_model_score, ppo_model_score)



=== Final Comparison ===
Base model: 10.00%
PPO model : 90.00%
Delta (PPO - Base): +80.00%
    PPO worked!
